# Gemma 3 1B Corporate Chatbot - QLoRA Fine-tuning (Colab)

This notebook is a thin wrapper around the scripts in this project. All the
actual logic (dataset validation, splitting, training, testing) lives in
`scripts/` and `src/corporate_chatbot/` - this notebook just calls those
scripts in order so you can run the whole pipeline on a Colab GPU runtime.

**Before running:** In Colab, go to `Runtime > Change runtime type` and
select a GPU (e.g. T4). QLoRA training requires a CUDA GPU.

Steps: install deps -> authenticate with Hugging Face -> get the project
onto the runtime -> validate dataset -> split dataset -> train -> test.

## 1. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 2. Get the project onto this runtime

If you're running this notebook from within a clone of the project already
(e.g. opened directly from your repo), skip this cell. Otherwise, clone your
repository (replace the URL below) or upload the project as a zip via the
Colab file browser and `%cd` into it.

In [ ]:
# Example (edit REPO_URL to your own fork/remote before running):
# REPO_URL = "https://github.com/<your-username>/gemma-corporate-chatbot.git"
# !git clone $REPO_URL
# %cd gemma-corporate-chatbot

## 3. Authenticate with Hugging Face

`google/gemma-3-1b-it` is a gated model. Accept the license at
https://huggingface.co/google/gemma-3-1b-it, then authenticate below. You
can either paste a token into the interactive login prompt (not stored in
the notebook), or set `HF_TOKEN` as a Colab secret and load it from there.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 4. Validate the dataset

Starts with the bundled demo dataset. Replace `data/raw/company_data.jsonl`
with your real data later and re-run from here.

In [ ]:
!python scripts/validate_dataset.py --input data/examples/corporate_examples.jsonl

## 5. Prepare and split the dataset

In [ ]:
!python scripts/prepare_dataset.py \
    --input data/examples/corporate_examples.jsonl \
    --output data/processed/dataset.jsonl

!python scripts/split_dataset.py \
    --input data/processed/dataset.jsonl \
    --output-dir data

## 6. Train

Uses `configs/training.yaml` and `configs/lora.yaml`. Edit those files (or
pass different `--config`/`--lora-config` paths) to change hyperparameters.

In [ ]:
!python scripts/train.py --config configs/training.yaml --lora-config configs/lora.yaml

## 7. Evaluate: base model vs fine-tuned model

In [ ]:
!python scripts/evaluate.py --adapter outputs/gemma3-1b-corporate-lora

## 8. Interactive test

`scripts/test_model.py` is interactive (reads from stdin in a loop), which
doesn't work well inside a notebook cell. Run it from a Colab terminal
instead (Colab Pro) or download the adapter and run it locally:

```bash
python scripts/test_model.py --adapter outputs/gemma3-1b-corporate-lora
```